In [2]:
# !pip install -q ipdb
# import ipdb

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
from torchvision.datasets import CIFAR10
from torchvision.transforms import Compose, ToTensor, Normalize
import matplotlib.pyplot as plt
from random import randint

In [ ]:
tr = Compose([
    ToTensor(),
    Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
train_dataset = CIFAR10('/.', train=True, download=True, transform=tr)
test_dataset = CIFAR10('/.', train=False, download=False, transform=tr)

clases = ('avión', 'auto', 'ave', 'gato', 'venado',
           'perro', 'rana', 'caballo', 'barco', 'camión')

In [ ]:
class InceptionModule(nn.Module):
  def __init__(self, 
               in_channels,
               n_classes, 
               ch_3x3_reduce=96, 
               ch_5x5_reduce=16,
               ch_3x3=128,
               ch_5x5=32,
               ch_pool_proj=32,
               ch_1x1=64
    ):
    super(InceptionModule, self).__init__()

    # Branch 1 
    self.conv_1p1_c1 = nn.Conv2d(in_channels, ch_3x3_reduce, (1,1), stride=1, padding=0)
    self.conv_3p3 = nn.Conv2d(ch_3x3_reduce, ch_3x3, (3,3), stride=1, padding=1)

    # Branch 2
    self.conv_1p1_c2 = nn.Conv2d(in_channels, ch_5x5_reduce, (1,1), stride=1, padding=0)
    self.conv_5p5 = nn.Conv2d(ch_5x5_reduce, ch_5x5, (5,5), stride=1, padding=2)

    # Branch 3
    self.pool = nn.MaxPool2d((3,3), stride=1, padding=1)
    self.conv_1p1_d3 = nn.Conv2d(in_channels, ch_pool_proj, (1,1), stride=1, padding=0)

    # Branch 4
    self.conv_1p1_d4 = nn.Conv2d(in_channels, ch_1x1, (1,1), stride=1, padding=0)

  def forward(self, x):
    # Branch 1
    x1_0 = self.conv_1p1_c1(x)
    x1 = self.conv_3p3(x1_0)

    # Branch 2
    x2_0 = self.conv_1p1_c2(x)
    x2 = self.conv_5p5(x2_0)

    # Branch 3
    x_pool = self.pool(x)
    x3 = self.conv_1p1_d3(x_pool)

    # Branch 4
    x4 = self.conv_1p1_d4(x)

    x = torch.cat([x1, x2, x3, x4], dim=1)  # (B, out_channels, H, W)

    return x

In [ ]:
class BasicConv2d(nn.Module):
    def __init__(self, in_channels, out_channels,**kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, **kwargs)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.conv(x))

In [ ]:
class GoogLeNet(nn.Module):
  def __init__(self, n_classes, use_aux_logits=True):
    super(GoogLeNet, self).__init__()

    # Define las capas de convolución y pooling de GoogLeNet
    self.stem = nn.Sequential(
        BasicConv2d(32, 32, kernel_size = 3, stride = 1 , padding = 1),
        nn.BatchNorm2d(32),
        BasicConv2d(32, 32, kernel_size = 1, stride = 1 , padding = 0),
        BasicConv2d(32, 30, kernel_size = 3, stride = 1 , padding = 0),
        nn.BatchNorm2d(30),
        nn.MaxPool2d(3, stride = 1) #(28x28)
    )

    # Decide si usar la clasificación auxiliar
    self.use_aux_logits = use_aux_logits
    if self.use_aux_logits: #...

In [ ]:

    # USEFUL?
    pooled = self.global_avg_pool(x)        # (B, out_channels, 1, 1)
    hidden = torch.flatten(pooled, 1)       # (B, out_channels)

    logits = self.fc(hidden)                # (B, n_classes)
     # logit/hidden 
    self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
    out_channels = ch_3x3 + ch_5x5 + ch_pool_proj + ch_1x1
    self.fc = nn.Linear(out_channels, n_classes)

, {
        "logits": logits,
        "hidden": hidden}

In [6]:
x = torch.randn(4, 3, 32, 32)

model = InceptionModule(
    in_channels=3,
    n_classes=10
)

out = model(x)

print(out["hidden"].shape)
print(out["logits"].shape)

torch.Size([4, 256])
torch.Size([4, 10])
